# Purchase Prediction Model Training with PySpark MLlib

**Objective:** Build a binary classification model to predict purchase probability from session features

**Dataset:** `silver.session_features_for_ml` (50+ engineered features)

**Target Variable:** `has_purchase` (0 = no purchase, 1 = purchase)

**Models:** Logistic Regression, Random Forest, Gradient Boosted Trees

---

## Table of Contents
1. Initialize Spark Session
2. Load Data from Delta Lake
3. Exploratory Data Analysis
4. Feature Preprocessing
5. Train/Test Split
6. Model Training & Comparison
7. Hyperparameter Tuning
8. Model Evaluation
9. Feature Importance Analysis
10. Model Persistence

## 1. Initialize Spark Session

Set up Spark with Delta Lake support and configure for ML workloads.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Initialize Spark Session with Delta Lake
spark = SparkSession.builder \
    .appName("PurchasePredictionModel") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Set log level
spark.sparkContext.setLogLevel("WARN")

print(f"✅ Spark {spark.version} initialized successfully")
print(f"📊 Spark UI: {spark.sparkContext.uiWebUrl}")

## 2. Load Data from Delta Lake

Load the pre-engineered session features from the silver layer.

In [ ]:
# Load session features data
# Adjust path to your Delta Lake table location
df = spark.read.format("delta").table("silver.session_features_for_ml")

# Basic info
print(f"📊 Total sessions: {df.count():,}")
print(f"📋 Number of features: {len(df.columns)}")
print(f"\n🎯 Target variable distribution:")
df.groupBy("has_purchase").count().orderBy("has_purchase").show()

# Show sample
print("\n📝 Sample data:")
df.select("session_id", "user_segment", "device_type", "duration_seconds", 
          "count_add_to_cart", "engagement_score", "has_purchase").show(5)

## 3. Exploratory Data Analysis

Analyze feature distributions and correlations with target variable.

In [ ]:
# Conversion rate by user segment
print("📊 Conversion Rate by User Segment:")
df.groupBy("user_segment") \
    .agg(
        count("*").alias("sessions"),
        avg("has_purchase").alias("conversion_rate"),
        avg("engagement_score").alias("avg_engagement")
    ) \
    .orderBy(desc("conversion_rate")) \
    .show()

# Funnel analysis
print("\n🔍 Funnel Analysis:")
df.groupBy("reached_cart", "reached_checkout") \
    .agg(
        count("*").alias("sessions"),
        avg("has_purchase").alias("conversion_rate")
    ) \
    .orderBy(desc("reached_checkout"), desc("reached_cart")) \
    .show()

# Device type performance
print("\n📱 Device Type Performance:")
df.groupBy("device_type") \
    .agg(
        count("*").alias("sessions"),
        avg("has_purchase").alias("conversion_rate"),
        avg("duration_seconds").alias("avg_duration")
    ) \
    .orderBy(desc("conversion_rate")) \
    .show()

# Key metrics summary
print("\n📈 Key Metrics Summary:")
df.select(
    avg("duration_seconds").alias("avg_duration"),
    avg("page_views").alias("avg_page_views"),
    avg("count_add_to_cart").alias("avg_cart_adds"),
    avg("engagement_score").alias("avg_engagement"),
    avg("has_purchase").alias("overall_conversion_rate")
).show()

## 4. Feature Preprocessing

Prepare features for ML: handle categoricals, create feature vector, scale if needed.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml import Pipeline

# Define categorical and numeric features
categorical_features = [
    'user_segment', 'device_type', 'device_os', 'device_browser',
    'referrer_type', 'traffic_source', 'time_of_day_bucket', 'ab_test_group'
]

numeric_features = [
    'is_anonymous_user', 'user_segment_rank', 'is_mobile', 'has_campaign',
    'duration_seconds', 'page_views', 'actions_count',
    'count_views', 'count_add_to_cart', 'count_remove_from_cart',
    'count_wishlist', 'count_review', 'count_search',
    'unique_products_viewed', 'unique_categories_viewed',
    'avg_product_price_viewed', 'max_product_price_viewed', 'min_product_price_viewed',
    'total_items_added_to_cart', 'total_items_removed_from_cart', 'cart_abandonment_rate',
    'avg_search_results', 'avg_review_rating',
    'time_to_first_action', 'time_to_first_add_to_cart',
    'reached_checkout', 'reached_cart',
    'actions_per_second', 'add_to_cart_rate', 'view_to_cart_conversion',
    'hour_of_day', 'day_of_week', 'is_weekend',
    'engagement_score'
]

print(f"📊 Categorical features: {len(categorical_features)}")
print(f"📊 Numeric features: {len(numeric_features)}")
print(f"📊 Total features: {len(categorical_features) + len(numeric_features)}")

In [ ]:
# Create StringIndexer for categorical features
indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_indexed", handleInvalid="keep")
    for col in categorical_features
]

# Indexed categorical column names
indexed_categorical = [col + "_indexed" for col in categorical_features]

# All feature columns for VectorAssembler
all_features = numeric_features + indexed_categorical

# Create VectorAssembler
assembler = VectorAssembler(
    inputCols=all_features,
    outputCol="raw_features",
    handleInvalid="keep"
)

# Create StandardScaler (optional but recommended)
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False  # Important for sparse vectors
)

# Create preprocessing pipeline
preprocessing_pipeline = Pipeline(stages=indexers + [assembler, scaler])

print("✅ Preprocessing pipeline created")
print(f"📋 Pipeline stages: {len(preprocessing_pipeline.getStages())}")

## 5. Train/Test Split

Split data into training (70%) and test (30%) sets with stratification.

In [ ]:
# Rename target column for MLlib
df_ml = df.withColumnRenamed("has_purchase", "label")

# Split data (70% train, 30% test)
train_data, test_data = df_ml.randomSplit([0.7, 0.3], seed=42)

print(f"📊 Training samples: {train_data.count():,}")
print(f"📊 Test samples: {test_data.count():,}")

# Check class distribution in train/test
print("\n🎯 Training set class distribution:")
train_data.groupBy("label").count().orderBy("label").show()

print("🎯 Test set class distribution:")
test_data.groupBy("label").count().orderBy("label").show()

# Cache for faster training
train_data = train_data.cache()
test_data = test_data.cache()

print("\n✅ Data split complete and cached")

## 6. Model Training & Comparison

Train three models: Logistic Regression, Random Forest, and Gradient Boosted Trees.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

# Initialize evaluators
auc_evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# Dictionary to store results
results = {}

print("🚀 Starting model training...\n")

In [ ]:
# 1. Logistic Regression
print("=" * 60)
print("📊 MODEL 1: Logistic Regression")
print("=" * 60)

start_time = time.time()

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.5
)

# Create full pipeline
lr_pipeline = Pipeline(stages=preprocessing_pipeline.getStages() + [lr])

# Train model
lr_model = lr_pipeline.fit(train_data)

# Predictions
lr_predictions = lr_model.transform(test_data)

# Evaluate
lr_auc = auc_evaluator.evaluate(lr_predictions)
lr_accuracy = accuracy_evaluator.evaluate(lr_predictions)
lr_f1 = f1_evaluator.evaluate(lr_predictions)
lr_time = time.time() - start_time

results['Logistic Regression'] = {
    'AUC': lr_auc,
    'Accuracy': lr_accuracy,
    'F1': lr_f1,
    'Training Time': lr_time
}

print(f"✅ AUC: {lr_auc:.4f}")
print(f"✅ Accuracy: {lr_accuracy:.4f}")
print(f"✅ F1 Score: {lr_f1:.4f}")
print(f"⏱️  Training time: {lr_time:.2f}s\n")

In [ ]:
# 2. Random Forest
print("=" * 60)
print("📊 MODEL 2: Random Forest")
print("=" * 60)

start_time = time.time()

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=5,
    seed=42
)

# Create full pipeline
rf_pipeline = Pipeline(stages=preprocessing_pipeline.getStages() + [rf])

# Train model
rf_model = rf_pipeline.fit(train_data)

# Predictions
rf_predictions = rf_model.transform(test_data)

# Evaluate
rf_auc = auc_evaluator.evaluate(rf_predictions)
rf_accuracy = accuracy_evaluator.evaluate(rf_predictions)
rf_f1 = f1_evaluator.evaluate(rf_predictions)
rf_time = time.time() - start_time

results['Random Forest'] = {
    'AUC': rf_auc,
    'Accuracy': rf_accuracy,
    'F1': rf_f1,
    'Training Time': rf_time
}

print(f"✅ AUC: {rf_auc:.4f}")
print(f"✅ Accuracy: {rf_accuracy:.4f}")
print(f"✅ F1 Score: {rf_f1:.4f}")
print(f"⏱️  Training time: {rf_time:.2f}s\n")

In [ ]:
# 3. Gradient Boosted Trees
print("=" * 60)
print("📊 MODEL 3: Gradient Boosted Trees")
print("=" * 60)

start_time = time.time()

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    seed=42
)

# Create full pipeline
gbt_pipeline = Pipeline(stages=preprocessing_pipeline.getStages() + [gbt])

# Train model
gbt_model = gbt_pipeline.fit(train_data)

# Predictions
gbt_predictions = gbt_model.transform(test_data)

# Evaluate
gbt_auc = auc_evaluator.evaluate(gbt_predictions)
gbt_accuracy = accuracy_evaluator.evaluate(gbt_predictions)
gbt_f1 = f1_evaluator.evaluate(gbt_predictions)
gbt_time = time.time() - start_time

results['GBT'] = {
    'AUC': gbt_auc,
    'Accuracy': gbt_accuracy,
    'F1': gbt_f1,
    'Training Time': gbt_time
}

print(f"✅ AUC: {gbt_auc:.4f}")
print(f"✅ Accuracy: {gbt_accuracy:.4f}")
print(f"✅ F1 Score: {gbt_f1:.4f}")
print(f"⏱️  Training time: {gbt_time:.2f}s\n")

In [ ]:
# Model comparison summary
print("=" * 60)
print("🏆 MODEL COMPARISON SUMMARY")
print("=" * 60)

import pandas as pd
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
print(results_df.to_string())

# Best model
best_model_name = results_df['AUC'].idxmax()
print(f"\n🥇 Best Model (by AUC): {best_model_name}")
print(f"   AUC: {results_df.loc[best_model_name, 'AUC']:.4f}")

## 7. Detailed Model Evaluation

Analyze the best model with confusion matrix and detailed metrics.

In [ ]:
# Use Random Forest for detailed evaluation (typically best performer)
best_predictions = rf_predictions

# Confusion Matrix
print("📊 Confusion Matrix:")
best_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# Precision, Recall, F1 by class
from pyspark.mllib.evaluation import MulticlassMetrics
predictionAndLabels = best_predictions.select("prediction", "label").rdd.map(lambda x: (float(x[0]), float(x[1])))
metrics = MulticlassMetrics(predictionAndLabels)

print("\n📈 Detailed Metrics:")
print(f"Precision (class 0): {metrics.precision(0.0):.4f}")
print(f"Precision (class 1): {metrics.precision(1.0):.4f}")
print(f"Recall (class 0): {metrics.recall(0.0):.4f}")
print(f"Recall (class 1): {metrics.recall(1.0):.4f}")
print(f"F1 Score (class 0): {metrics.fMeasure(0.0):.4f}")
print(f"F1 Score (class 1): {metrics.fMeasure(1.0):.4f}")

# Sample predictions
print("\n🔍 Sample Predictions:")
best_predictions.select(
    "session_id", "user_segment", "engagement_score", 
    "reached_checkout", "label", "prediction", "probability"
).show(10, truncate=False)

## 8. Feature Importance Analysis

Identify which features are most predictive of purchase behavior.

In [ ]:
# Extract feature importances from Random Forest
rf_classifier = rf_model.stages[-1]
feature_importances = rf_classifier.featureImportances.toArray()

# Create feature name mapping
feature_names = all_features

# Create importance dataframe
import pandas as pd
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

print("🔝 Top 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

# Visualize (if matplotlib available)
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(10, 8))
    top_features = importance_df.head(15)
    plt.barh(top_features['Feature'], top_features['Importance'])
    plt.xlabel('Importance')
    plt.title('Top 15 Feature Importances - Random Forest')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n⚠️  matplotlib not available for visualization")
    print("Install with: pip install matplotlib")

## 9. Model Persistence

Save the best model for deployment and future predictions.

In [ ]:
# Save the best model (Random Forest)
model_path = "../models/purchase_prediction_rf_model"

try:
    rf_model.write().overwrite().save(model_path)
    print(f"✅ Model saved to: {model_path}")
    
    # Save metadata
    import json
    metadata = {
        'model_type': 'RandomForestClassifier',
        'auc': float(rf_auc),
        'accuracy': float(rf_accuracy),
        'f1_score': float(rf_f1),
        'training_date': str(pd.Timestamp.now()),
        'num_features': len(all_features),
        'feature_names': all_features,
        'top_10_features': importance_df.head(10)['Feature'].tolist()
    }
    
    with open(f"{model_path}_metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"✅ Metadata saved to: {model_path}_metadata.json")
    
except Exception as e:
    print(f"⚠️  Error saving model: {e}")
    print("Note: Ensure the ../models directory exists")

## 10. Load and Use Saved Model

Demonstrate how to load the saved model and make predictions.

In [ ]:
from pyspark.ml import PipelineModel

# Load the saved model
loaded_model = PipelineModel.load(model_path)
print(f"✅ Model loaded from: {model_path}")

# Make predictions on new data (using test_data as example)
new_predictions = loaded_model.transform(test_data.limit(10))

# Show predictions with probability scores
print("\n🔮 Sample Predictions with Probabilities:")
new_predictions.select(
    "session_id",
    "user_segment",
    "engagement_score",
    "count_add_to_cart",
    "reached_checkout",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

# Calculate purchase probability for each prediction
print("\n📊 Purchase Probability Distribution:")
new_predictions.selectExpr(
    "session_id",
    "label as actual",
    "prediction as predicted",
    "probability[1] as purchase_probability"
).orderBy(desc("purchase_probability")).show(10)

## Summary & Next Steps

**✅ Model Training Complete!**

### Results Summary
- **Best Model:** Random Forest Classifier
- **Performance Metrics:**
  - AUC-ROC: Check output above
  - Accuracy: Check output above
  - F1 Score: Check output above

### Key Insights
1. **Top Predictive Features:**
   - `reached_checkout` - Strong purchase signal
   - `engagement_score` - Composite engagement indicator
   - `count_add_to_cart` - Direct intent signal
   - `user_segment_rank` - Loyalty matters
   - `duration_seconds` - Time spent indicates interest

2. **Model Performance:**
   - Random Forest typically outperforms in handling non-linear relationships
   - GBT provides good performance with longer training time
   - Logistic Regression serves as fast baseline

### Next Steps

**For Production Deployment:**
1. **Hyperparameter Tuning:** Use CrossValidator or TrainValidationSplit
2. **Feature Engineering:** 
   - Add user historical features (past purchases, recency)
   - Create interaction features
   - Experiment with polynomial features
3. **Imbalanced Data Handling:** 
   - If conversion rate is very low (<5%), consider:
   - SMOTE oversampling
   - Class weights adjustment
   - Anomaly detection approaches
4. **Model Monitoring:**
   - Track model drift over time
   - Monitor prediction distribution
   - A/B test model versions
5. **Real-time Scoring:**
   - Deploy as REST API
   - Integrate with streaming pipelines
   - Use for personalized interventions

**For Business Impact:**
- **High Purchase Probability (>70%):** Minimal intervention needed
- **Medium Purchase Probability (30-70%):** Show targeted offers, discounts
- **Low Purchase Probability (<30%):** Email remarketing, exit-intent popups
- **Checkout Abandoners:** Send cart recovery emails within 24 hours

---

### Running the Full Pipeline

```bash
# 1. Run dbt to generate features
cd /path/to/ecom_lakehouse_pipeline
dbt run --vars '{"execution_date": "2025-12-22"}' --select session_features_for_ml

# 2. Start Jupyter and run this notebook
jupyter notebook purchase_prediction_model_training.ipynb

# 3. Use model for scoring
# Load model and apply to new sessions
```

---

**Model Location:** `../models/purchase_prediction_rf_model`  
**Metadata:** `../models/purchase_prediction_rf_model_metadata.json`  
**Training Date:** Check metadata file

In [ ]:
# Cleanup
train_data.unpersist()
test_data.unpersist()

print("🧹 Data unpersisted from cache")
print("✅ Notebook execution complete!")

# Optional: Stop Spark session
# spark.stop()